In [1]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 4.8 MB/s eta 0:00:00


In [2]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [3]:
import os
from pyngrok import ngrok

In [3]:
ngrok.kill()

NameError: name 'ngrok' is not defined

In [4]:
import requests
from pyngrok import ngrok

# Terminate any existing ngrok tunnels to prevent 'tunnel already exists' error
ngrok.kill()

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://spending-chastise-bullion.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://spending-chastise-bullion.ngrok-free.dev


True

In [5]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        response_modalities=["TEXT"],
    )
)

In [6]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [7]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學 (Ming Hsin University of Science and Technology)，簡稱明新科大，是一所位於台灣**新竹縣新豐鄉**的知名科技大學。

以下是明新科技大學的簡要介紹：

1.  **創校歷史與發展**：
    *   學校創立於**1966年**，最初為「明新工業專科學校」。
    *   經過數十年的發展與耕耘，於**2002年升格為科技大學**，擁有悠久的辦學歷史和豐富的技職教育經驗。

2.  **教育理念與特色**：
    *   校訓為「**誠樸精勤**」，秉持技職教育務實致用的精神。
    *   致力於培養具備**專業技能、實作能力、創新思維及國際視野**的產業人才。
    *   特別強調**理論與實務結合**，課程設計與產業需求接軌，鼓勵學生取得專業證照，提升職場競爭力。

3.  **學院設置與學術領域**：
    *   學校設有多元學院，主要包括：
        *   **工程學院**
        *   **管理學院**
        *   **服務產業學院**
        *   **設計學院**
    *   提供學士、碩士等多層次的學術與專業訓練，涵蓋了工程、資訊、管理、商業、設計、服務休閒等多元領域。

4.  **地理位置優勢與產學合作**：
    *   明新科大位於新竹縣，毗鄰**新竹科學園區**及周邊工業區，享有得天獨厚的地理優勢。
    *   學校積極推動**產學合作**，與眾多高科技企業及產業建立緊密關係，為學生提供豐富的實習、就業與實務專題機會。這使得畢業生在就業市場上具有較強的競爭力。

5.  **發展重點**：
    *   近年來，學校在**創新創業、綠能科技、智慧製造、物聯網、大數據、休閒管理、設計文創**等領域發展特色，並積極推動國際交流與合作。

總體而言，明新科技大學是一所歷史悠久、務實創新，且與產業緊密結合的科技大學，長期以來為台灣及國際社會培育了眾多優秀的專業技術與管理人才。


In [8]:
result2 = stateful_query("校長是誰？")
print(result2)

截至目前 (2024年)，明新科技大學的校長是**劉國偉**博士。


In [9]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit


BODY:  {"destination":"Ua5551030dc64321324b89c4f0ea9257d","events":[{"type":"message","message":{"type":"text","id":"616372823820599916","quoteToken":"wIJzepcq2-nPssEFnQN3vX0bI1bd3hWa1kWM6gu32CrDmdif-bwnUtsCWsZx7M1nJRY49iACttnjdWGq0Li8jbdvbTT9gmb-8aAEdzPwNMrJCGaTTcfEAP0on9Pp9xDhZ54En1LvJ1PEIccm4spkoQ","markAsReadToken":"ul_XuC_dvmYSlB08BsTTGk3TF9KFQltHxv5BGVhKI2WCeiPOlpRmESKjuvegSVEwFtVOsMWUPLRhKpR_jcJp2bXXz5D6sfAzx-v3Al4cLpStxq33Rg9t-NBPzDVmVz0yj5NgMYUqZgm9ytuDNEJVbvaEx5MCMZqk_tP_bi8zIhYVJV6gMSzqBoZCepWbEsRGCBTVLvV0bZZSTBoMrFdaJQ","text":"AI  簡介明新科技大學"},"webhookEventId":"01KSYMA1SB7J82K7AQDXPC69AQ","deliveryContext":{"isRedelivery":false},"timestamp":1780218136319,"source":{"type":"user","userId":"Ue65570520db252700cb187ff02c70937"},"replyToken":"ac645d4b7ab94501b06b7708fa8bdaae","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [31/May/2026 09:02:29] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"Ua5551030dc64321324b89c4f0ea9257d","events":[{"type":"message","message":{"type":"text","id":"616372849791729754","quoteToken":"yM44o1s1LmEbd4ekva87UMPBnYFcQXIh4hR95Gc5uEGGtEO37j_SO4TArWxlUtlttrp1G4Ac-VQ43d84qs3tUgNTqL_f2IPmlqpEculb4gfrXtjxXZM4taAiZCU5KjpqK8GpMbjcEgDpdxos5AAQIg","markAsReadToken":"K2tywAXF0354adKRLxwIcz_h2_Bw3AEaXkCZJ-iO-UcNato2AWIkPPe3jgKwnA5hZ1Eh6wU2oZZT0kjwZDb5uBh_0xy6_APGJNbd3qU02UYxP0kDan1UzaOCVO0N48qYtFrXHisv_Vj9UFq1_5DunsFo19fzam-SMkYbRpCMrzFn4yQtD6Y9ZH6wckU4w2f1ulps7f3Q4bbOXtuDPLOLkA","text":"AI  校長是誰"},"webhookEventId":"01KSYMAHAAS777DRJZT9YXDTPN","deliveryContext":{"isRedelivery":false},"timestamp":1780218151757,"source":{"type":"user","userId":"Ue65570520db252700cb187ff02c70937"},"replyToken":"e84c6d99358544fda9b3955d2fbafa04","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [31/May/2026 09:02:33] "POST / HTTP/1.1" 200 -
